In [3]:
!nvidia-smi

Sat Jul 25 01:42:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   27C    P0             58W /  400W |    5997MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import os
import torch
import traceback
import time

# 1. IMPORT CORE UTILITIES
import core_config
from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

# 2. IMPORT ML LIBRARIES
from diffusers import AutoPipelineForText2Video
from diffusers.utils import export_to_video

# ==============================================================================
# PIPELINE: WAN 2.1 (14B)
# Target: High-fidelity cinematic environments, complex motion, drone shots.
# Requirements: Python 3.9, PyTorch 2.1.0, requires ~28GB VRAM (A100 compatible).
# ==============================================================================

class WanVideoWorker:
    def __init__(self, gpu_id=None):
        self.gpu_id = self._select_gpu() if gpu_id is None else str(gpu_id)
        os.environ["CUDA_VISIBLE_DEVICES"] = self.gpu_id
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.pipe = None

    @staticmethod
    def _select_gpu():
        import subprocess
        try:
            raw = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
                encoding="utf-8",
            )
            free_mb = [int(x.strip()) for x in raw.splitlines() if x.strip()]
            if not free_mb:
                raise RuntimeError("nvidia-smi returned no GPUs")
            gpu = str(free_mb.index(max(free_mb)))
            print(f"[WAN] Auto-selected GPU {gpu} ({max(free_mb)} MB free)")
            return gpu
        except Exception as exc:
            raise RuntimeError(f"Could not select an available GPU: {exc}") from exc
        
    def load_model(self):
        print(f"[WAN-WORKER-GPU{self.gpu_id}] Verifying Wan2.1-T2V-14B in local cache...")
        kwargs = dict(torch_dtype=torch.bfloat16)
        try:
            self.pipe = AutoPipelineForText2Video.from_pretrained(
                "Wan-AI/Wan2.1-T2V-14B",
                local_files_only=True,
                **kwargs,
            ).to(self.device)
            print(f"[WAN-WORKER-GPU{self.gpu_id}] Cache hit; no model download required.")
        except Exception:
            required_gb = (
                estimate_hf_repo_bytes("Wan-AI/Wan2.1-T2V-14B") / BYTES_PER_GB
                + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
            )
            print(
                f"[WAN-WORKER-GPU{self.gpu_id}] Cache incomplete. "
                f"Storage admission budget: {required_gb:.2f} GB"
            )
            with managed_storage_operation(
                "models",
                required_gb=required_gb,
                require_fresh_quota=True,
                minimum_free_gb=float(
                    os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")
                ),
            ):
                self.pipe = AutoPipelineForText2Video.from_pretrained(
                    "Wan-AI/Wan2.1-T2V-14B",
                    **kwargs,
                ).to(self.device)

        self.pipe.enable_model_cpu_offload()
        print(f"[WAN-WORKER-GPU{self.gpu_id}] Model loaded and ready.")

    def generate(self, prompt, num_frames=81, fps=16):
        if not self.pipe:
            self.load_model()
            
        print(f"[WAN-WORKER-GPU{self.gpu_id}] Generating: '{prompt}'")
        try:
            video_frames = self.pipe(
                prompt=prompt, 
                num_frames=num_frames
            ).frames[0]
            
            timestamp = int(time.time())
            filename = f"wan_vid_{timestamp}.mp4"
            output_path = os.path.join(SYSTEM_DIRS["outputs"], "videos", filename)

            estimated_gb = max(
                0.05,
                len(video_frames) * 100 * 1_000_000 / 8 / BYTES_PER_GB * 2.0 / max(1, fps),
            )
            with managed_storage_operation(
                "outputs",
                required_gb=estimated_gb,
                require_fresh_quota=False,
                minimum_free_gb=0.05,
            ):
                export_to_video(video_frames, output_path, fps=fps)

            print(f"[WAN-WORKER-GPU{self.gpu_id}] Success! Saved to: {output_path}")
            return output_path
            
        except Exception as e:
            print(f"[WAN-WORKER-GPU{self.gpu_id}] Generation failed:")
            traceback.print_exc()
            return None

# Execution Block
if __name__ == "__main__":
    worker = WanVideoWorker(gpu_id=None)
    worker.generate("Cinematic drone shot over a glowing bioluminescent forest at night, 4k resolution.")

[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env


AttributeError: module 'torch' has no attribute 'xpu'